In [1]:
import pandas as pd
import numpy as np

master = pd.read_csv('player.csv')
batting = pd.read_csv('batting.csv')

master['Player'] = master['name_first'] + ' ' + master['name_last']

data = batting.merge(master[['player_id', 'Player']], on='player_id', how='left')

data.head()

,player_id,year,stint,team_id,league_id,g,ab,r,h,double,...,sb,cs,bb,so,ibb,hbp,sh,sf,g_idp,Player
0,abercda01,1871,1,TRO,NaN,1,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Frank Abercrombie
1,addybo01,1871,1,RC1,NaN,25,118.0,30.0,32.0,6.0,...,8.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN,Bob Addy
2,allisar01,1871,1,CL1,NaN,29,137.0,28.0,40.0,4.0,...,3.0,1.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN,Art Allison
3,allisdo01,1871,1,WS3,NaN,27,133.0,28.0,44.0,10.0,...,1.0,1.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,Doug Allison
4,ansonca01,1871,1,RC1,NaN,25,120.0,29.0,39.0,11.0,...,6.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,Cap Anson


In [2]:
DATASET_NAME = "baseball-database"

NUMERIC_COLUMNS = [
    "stint", "g", "ab", "r", "h", "double", "triple", "hr", "rbi", "sb",
    "cs", "bb", "so", "ibb", "hbp", "sh", "sf", "g_idp"
]


def clean_mlb_data(data):
    data["year"] = pd.to_numeric(data["year"], errors="coerce")
    data = data.dropna(subset=["year"]).copy()
    data["year"] = data["year"].astype(int)

    for column in NUMERIC_COLUMNS:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    return data


data = clean_mlb_data(data)

print(f"Loaded {len(data):,} rows from {DATASET_NAME}")
print(f"Players: {data['Player'].nunique():,}")
print(f"Seasons: {data['year'].min()}-{data['year'].max()}")

data.head()

Loaded 101,332 rows from baseball-database
Players: 18,023
Seasons: 1871-2015


,player_id,year,stint,team_id,league_id,g,ab,r,h,double,...,sb,cs,bb,so,ibb,hbp,sh,sf,g_idp,Player
0,abercda01,1871,1,TRO,NaN,1,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,Frank Abercrombie
1,addybo01,1871,1,RC1,NaN,25,118.0,30.0,32.0,6.0,...,8.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN,Bob Addy
2,allisar01,1871,1,CL1,NaN,29,137.0,28.0,40.0,4.0,...,3.0,1.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN,Art Allison
3,allisdo01,1871,1,WS3,NaN,27,133.0,28.0,44.0,10.0,...,1.0,1.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,Doug Allison
4,ansonca01,1871,1,RC1,NaN,25,120.0,29.0,39.0,11.0,...,6.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,Cap Anson


In [ ]:
from google import genai
from google.genai import types

API_KEY = None  # Add your API key here
MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=API_KEY)
print(f"Client ready for {MODEL}.")

Client ready for gemini-2.5-flash.


In [4]:
# This is the smallest useful Gemini request in the notebook.
intro_response = client.models.generate_content(
    model=MODEL,
    contents="In 2 sentences, explain what an API does for a beginner."
)

print(intro_response.text)

An API (Application Programming Interface) acts like a messenger, allowing different software programs to talk to each other. It lets one program request information or trigger actions from another program, without needing to know all the complex details of how the other program actually works internally.


In [5]:
from textwrap import dedent

DATA_DICTIONARY = dedent(
    """
    DataFrame name: data
    Rows: about 101,332
    Meaning of each row: One row per MLB player per season per team stint (a player traded mid-season appears twice).
    Time span: 1871 through 2015

    Important identity columns
    - year: MLB season year
    - Player: full player name after merging with master.csv
    - stint: player's stint (order of appearances within a season)
    - player_id: Unique player identifier (not human-readable)
    - team_id: team abbreviation
    - league_id: League: AL (American) or NL (National)

    Important counting stats
    - g: games played
    - ab: at-bats (official plate appearances, excluding walks, and hit by pitch)
    - r: runs scored
    - h: hits (total)
    - double: doubles
    - triple: triples
    - hr: home runs
    - rbi: runs batted in
    - sb: stolen bases
    - cs: caught stealing
    - bb: walks (base on balls)
    - so: strikeouts
    - ibb: intentional walks
    - hbp: hit by pitch
    - sh: sacrifice hits (bunts)
    - sf: sacrifice flies (non-bunts)
    - g_idp: grounded into double play

    The following stats are not stored in the baseball-database (data) - they must be computed:
    - Batting Average (BA) = h / ab
    - On-Base Percentage (OBP) = (h + bb + hbp) / (ab + bb + hbp + sf)
    - Slugging Percentage (SLG) = (h + double + 2*triple + 3*hr) / ab
    - On-Base Plus Slugging (OPS) = obp + slg
    - Home runs per game = hr / g
    - Strikeout rate = so / ab
    - Singles = h - double - triple - hr

   Additional notes to keep in mind when working with this dataset:
    - Stats are season totals that you divide by ab or g depending on the stat - not just g. For example,
    batting average is shown above as h/ab not h/g.

    Useful query habits
    - A player traded mid-season appears twice (once per team). Check what value appears for traded players in the team_id
    column of your actual data and document it.
    - For player searches, use str.contains(..., case=False, na=False).
    - For career per-game leaderboards, create a per-game column first, then group by Player and average it.

    Dataset warnings
    - Post-2015 players are not in this file.
    - Some columns are NaN before the stat started being recorded.
    - A player traded mid-season appears twice (once per team). The stint column indicates the order of appearances within a season.
    For example, a player traded mid-season might have stint 1 with team_id "BOS" and stint 2 with team_id "LAD". This indicates
    the player started the season with Boston and was traded to Los Angeles mid-season. Make sure to total a players statistics across
    stints when doing career leaderboards.
    - Some players, because of their position (e.g. pitchers), may have very few at-bats (ab) but many
    games (g). This can be misleading when answering questions about "best" players. Because of this,
    always apply the following minimum thresholds when doing career leaderboards:
    
      - Batting average (ba): minimum 3,000 at-bats (ab)
      - On-base percentage (obp) / slugging (slg) / OPS: minimum 3,000 plate appearances
      - Home runs, RBIs, hits, runs: minimum 1,000 games (g)
      - Any rate stat (per game, per at-bat, etc.): minimum 1,000 games (g)
    
    These thresholds ensure that leaderboard answers reflect players with a substantial body of work,
    not outliers with small sample sizes. For example, a pitcher who went 2-for-4 over 10 seasons
    should not appear on a career batting average leaderboard above Ty Cobb.    
    """
).strip()

print(f"Data dictionary ready ({len(DATA_DICTIONARY.split())} words).")

Data dictionary ready (567 words).


In [12]:
import traceback

QUESTION_RULES = dedent(
    """
    You are helping a beginner explore MLB stats with pandas.

    Use the `run_pandas_query` tool exactly once before answering the question.
    The dataframe is ALWAYS named `df`. Never use `data`, `df2`, or any other name.
    Correct: df[df['ab'] >= 400]
    Incorrect: data[data['ab'] >= 400]
    Assign the final answer object to a variable named result.

    Rules for the tool code:
    - Write 1 to 6 lines of executable Python.
    - Stats are season totals that you divide by ab or g depending on the stat - not just g.
    - For season leaderboards, total up a player's stats across stints to avoid double-counting traded players.
    - If the tool returns an error, explain the error instead of guessing.
    - Final answer style: conversational, specific, and short.
    """
).strip()


def build_question_prompt(question: str) -> str:
    """Build the full prompt sent to Gemini for an MLB question.

    Args:
        question: The natural-language question the student wants to ask about
            the MLB dataset.

    Returns:
        A single prompt string containing the tool rules, the data dictionary,
        and the student's question.

    Example:
        prompt = build_question_prompt("Who led the MLB in home runs in 1990?")
        print(prompt[:400])
    """
    return dedent(
        f"""
        {QUESTION_RULES}

        Dataset description:
        {DATA_DICTIONARY}

        Student question: {question}
        """
    ).strip()


def show_prompt(question: str) -> None:
    """Print the exact prompt that will be sent to Gemini.

    Args:
        question: The question you plan to pass to `ask(...)`.

    Returns:
        None. This function prints text so you can inspect the prompt.

    Example:
        show_prompt("Which players had the highest career home runs per season?")
    """
    print(build_question_prompt(question))


QUERY_FUNCTION = types.FunctionDeclaration(
    name="run_pandas_query",
    description="Run short Python/pandas code against df and return a compact preview of the result.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": (
                    "Executable Python that uses only df, pd, and np. "
                    "Assign the final object to a variable named result."
                ),
            }
        },
        "required": ["code"],
    },
)

QUERY_TOOL = types.Tool(function_declarations=[QUERY_FUNCTION])


def clean_model_code(code: str) -> str:
    """Remove Markdown code fences from model-generated Python.

    Args:
        code: Raw text returned by Gemini. Sometimes the model wraps Python in
            triple backticks before sending it through the tool.

    Returns:
        A cleaned Python string that can be passed to `exec()`.

    Example:
        clean_model_code("```python\nresult = df.head()\n```")
    """
    code = code.strip()
    if code.startswith("```"):
        lines = code.splitlines()
        code = "\n".join(lines[1:])
        if code.endswith("```"):
            code = "\n".join(code.splitlines()[:-1])
    return code.strip()


def format_result_for_model(result) -> str:
    """Convert a pandas result into a short text preview for Gemini.

    Args:
        result: Usually a pandas DataFrame, pandas Series, or a scalar value
            produced by model-written pandas code.

    Returns:
        A compact text summary that is short enough to send back to Gemini in a
        follow-up API call.

    Example:
        sample = df[["Player", "r"]].head(3)
        print(format_result_for_model(sample))
    """
    if isinstance(result, pd.DataFrame):
        preview = result.head(20).to_string(index=False)
        return f"Result ({len(result)} rows):\n{preview}"

    if isinstance(result, pd.Series):
        preview = result.head(20).to_string()
        return f"Result:\n{preview}"

    return f"Result: {result}"


def run_pandas_query(code: str, debug: bool = False) -> dict:
    """Run model-generated pandas code against the local DataFrame.

    Args:
        code: Short Python code that uses only `df`, `pd`, and `np`, and stores
            the final answer in a variable named `result`.
        debug: If True, print the generated code and any Python traceback.

    Returns:
        A dictionary with `status`, `code`, and `data_context` fields. This is
        the payload sent back to Gemini after the tool executes.

    Example:
        run_pandas_query(
            "result = df[df['Player'].str.contains('Bonds', case=False, na=False)].head()",
            debug=True,
        )

    Warning:
        This classroom demo uses `exec()`, which can run arbitrary Python code.
        That is convenient for teaching, but unsafe in production.
    """
    code = clean_model_code(code)
    local_vars = {"df": data, "pd": pd, "np": np}

    if debug:
        print("-- Generated pandas code --")
        print(code)
        print()

    try:
        exec(code, {}, local_vars)
        result = local_vars.get("result")

        if result is None:
            return {
                "status": "error",
                "code": code,
                "data_context": "The code ran, but it did not create a variable named `result`.",
            }

        return {
            "status": "ok",
            "code": code,
            "data_context": format_result_for_model(result),
        }

    except Exception as error:
        if debug:
            traceback.print_exc()

        return {
            "status": "error",
            "code": code,
            "data_context": f"Code failed with error: {error}",
        }


FIRST_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=["run_pandas_query"],
        )
    ),
    temperature=0,
)

FINAL_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="NONE")
    ),
    temperature=0.3,
)


def ask(question: str, debug: bool = False) -> None:
    """Ask Gemini a question about the MLB dataset.

    Args:
        question: A natural-language question such as "Who were the top base stealers
            in 1988?" or "Compare Bonds and Ortiz as home run leaders."
        debug: If True, print the pandas code Gemini generated before the final
            explanation is shown.

    Returns:
        None. The function prints Gemini's final explanation to the notebook.

    Example:
        ask("Which players had the highest career home runs per season?", debug=True)
        ask("Who had the most home runs in 1990?")
    """
    prompt = build_question_prompt(question)
    conversation = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        )
    ]

    first_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FIRST_PASS_CONFIG,
    )

    if not first_response.function_calls:
        print(first_response.text)
        return

    tool_call = first_response.function_calls[0]
    tool_code = tool_call.args.get("code", "")
    tool_result = run_pandas_query(tool_code, debug=debug)

    conversation.append(first_response.candidates[0].content)
    conversation.append(
        types.Content(
            role="tool",
            parts=[
                types.Part.from_function_response(
                    name=tool_call.name,
                    response=tool_result,
                )
            ],
        )
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FINAL_PASS_CONFIG,
    )

    print(final_response.text)


print("Ready. Try ask('Which players had the highest career home runs per season?')")


Ready. Try ask('Which players had the highest career home runs per season?')


In [13]:
show_prompt("Which players had the highest career home runs per season?")

You are helping a beginner explore MLB stats with pandas.

Use the `run_pandas_query` tool exactly once before answering the question.
The dataframe is ALWAYS named `df`. Never use `data`, `df2`, or any other name.
Correct: df[df['ab'] >= 400]
Incorrect: data[data['ab'] >= 400]
Assign the final answer object to a variable named result.

Rules for the tool code:
- Write 1 to 6 lines of executable Python.
- Stats are season totals that you divide by ab or g depending on the stat - not just g.
- For season leaderboards, total up a player's stats across stints to avoid double-counting traded players.
- If the tool returns an error, explain the error instead of guessing.
- Final answer style: conversational, specific, and short.

        Dataset description:
        DataFrame name: data
 Rows: about 101,332
 Meaning of each row: One row per MLB player per season per team stint (a player traded mid-season appears twice).
 Time span: 1871 through 2015

 Important identity columns
 - year: MLB

In [33]:
ask("Who had the highest career batting average (minimum 10 seasons)?", debug=True)

-- Generated pandas code --
career_stats = df.groupby(['player_id', 'Player']).agg(career_h=('h', 'sum'), career_ab=('ab', 'sum'), seasons=('year', 'nunique')).reset_index()
career_stats['BA'] = career_stats['career_h'] / career_stats['career_ab']
filtered_stats = career_stats[(career_stats['career_ab'] >= 3000) & (career_stats['seasons'] >= 10)]
result = filtered_stats.sort_values(by='BA', ascending=False).head(1)

Ty Cobb had the highest career batting average with 0.366, playing for 24 seasons and accumulating 11,434 at-bats.


In [34]:
ask("What were Babe Ruth's stats season by season", debug=True)

-- Generated pandas code --
babe_ruth_df = df[df['Player'].str.contains('Babe Ruth', case=False, na=False)].groupby(['year', 'Player']).sum().reset_index()
babe_ruth_df['BA'] = babe_ruth_df['h'] / babe_ruth_df['ab']
babe_ruth_df['OBP'] = (babe_ruth_df['h'] + babe_ruth_df['bb'] + babe_ruth_df['hbp']) / (babe_ruth_df['ab'] + babe_ruth_df['bb'] + babe_ruth_df['hbp'] + babe_ruth_df['sf'])
babe_ruth_df['SLG'] = (babe_ruth_df['h'] + babe_ruth_df['double'] + 2 * babe_ruth_df['triple'] + 3 * babe_ruth_df['hr']) / babe_ruth_df['ab']
babe_ruth_df['OPS'] = babe_ruth_df['OBP'] + babe_ruth_df['SLG']
babe_ruth_df['Singles'] = babe_ruth_df['h'] - babe_ruth_df['double'] - babe_ruth_df['triple'] - babe_ruth_df['hr']
result = babe_ruth_df[['year', 'g', 'ab', 'r', 'h', 'Singles', 'double', 'triple', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'BA', 'OBP', 'SLG', 'OPS']].round(3)

Here are Babe Ruth's season-by-season stats, including his batting average (BA), on-base percentage (OBP), slugging percentage (SLG),

In [35]:
ask("Who hit the most home runs in the 1998 season?", debug=True)

-- Generated pandas code --
result = df[df['year'] == 1998].groupby('Player')['hr'].sum().nlargest(1).index[0]

In the 1998 season, Mark McGwire hit the most home runs.



In [8]:
ask("Compare Barry Bonds and Hank Aaron?", debug=True)

-- Generated pandas code --
result = df[df['Player'].isin(['Barry Bonds', 'Hank Aaron'])].groupby('Player')[['ab', 'h', 'double', 'triple', 'hr', 'bb', 'hbp', 'sf', 'rbi', 'r', 'g']].sum().assign(
    ba=lambda x: x['h'] / x['ab'],
    obp=lambda x: (x['h'] + x['bb'] + x['hbp']) / (x['ab'] + x['bb'] + x['hbp'] + x['sf']),
    slg=lambda x: (x['h'] + x['double'] + 2 * x['triple'] + 3 * x['hr']) / x['ab'],
    ops=lambda x: x['obp'] + x['slg']
)[['g', 'ab', 'h', 'hr', 'rbi', 'r', 'ba', 'obp', 'slg', 'ops']].round(3)

Here's a comparison of Barry Bonds and Hank Aaron's career stats:

| Player      | Games (g) | At-Bats (ab) | Hits (h) | Home Runs (hr) | RBIs (rbi) | Runs (r) | Batting Average (ba) | On-Base Percentage (obp) | Slugging Percentage (slg) | OPS (ops) |
|:------------|----------:|-------------:|---------:|---------------:|-----------:|---------:|---------------------:|-------------------------:|--------------------------:|----------:|
| Barry Bonds |    2986.0 |       9847.0 |

In [9]:
ask("Who had the most strikeouts per at-bat in their career?", debug=True)

-- Generated pandas code --
result = df.groupby(['player_id', 'Player']).agg(career_so=('so', 'sum'), career_ab=('ab', 'sum'), career_g=('g', 'sum')).reset_index()
result = result[result['career_g'] >= 1000]
result['career_so_rate'] = result['career_so'] / result['career_ab']
result = result.sort_values(by='career_so_rate', ascending=False).head(1)

LaTroy Hawkins had the highest career strikeout rate, with 0.875 strikeouts per at-bat. It's worth noting that as a pitcher, he had very few at-bats, which can lead to an extremely high rate.


In [14]:
ask("Which players had the best OPS in seasons where they had at least 400 at-bats?", debug=True)

-- Generated pandas code --
df_filtered = df[df['ab'] >= 400]
df_filtered['obp'] = (df_filtered['h'] + df_filtered['bb'] + df_filtered['hbp']) / (df_filtered['ab'] + df_filtered['bb'] + df_filtered['hbp'] + df_filtered['sf'])
df_filtered['slg'] = (df_filtered['h'] + df_filtered['double'] + 2 * df_filtered['triple'] + 3 * df_filtered['hr']) / df_filtered['ab']
df_filtered['ops'] = df_filtered['obp'] + df_filtered['slg']
result = df_filtered.sort_values(by='ops', ascending=False)[['Player', 'year', 'ops']].head(10)

In seasons with at least 400 at-bats, Barry Bonds had the best OPS in 2002 (1.381) and 2001 (1.379). Ted Williams in 1957 (1.257) and Mark McGwire in 1998 (1.222) also had exceptionally high OPS values.
